### HyDE - 

In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate
from langchain_community.document_loaders import WikipediaLoader

### Initialization of Embedding & LLM model

In [2]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini")

In [10]:
import wikipedia
_original_set_lang = wikipedia.set_lang

def _set_lang_https(lang):
    _original_set_lang(lang)
    wikipedia.API_URL = f"https://{lang}.wikipedia.org/w/api.php"

wikipedia.set_lang = _set_lang_https
wikipedia.set_user_agent("RAGAI/1.0 (contact: ajysharma704@gmail.com.com)")

wiki_docs_loader = WikipediaLoader(
    query="Bill Gates",
    load_max_docs=5,
)
docs=wiki_docs_loader.load()
docs

[Document(metadata={'title': 'Bill Gates', 'summary': "William Henry Gates III  (born October 28, 1955) is an American businessman and philanthropist. A pioneer of the microcomputer revolution of the 1970s and 1980s, he co-founded the software company Microsoft in 1975 with his childhood friend Paul Allen. Following Microsoft's initial public offering in 1986 and the subsequent increase in its stock price, Gates became the world's then-youngest billionaire in 1987, at age 31. Forbes magazine ranked him as the world's wealthiest person in their The World's Billionaires list for 18 out of 24 years between 1995 and 2017, including 13 years consecutively from 1995 to 2007. Gates became the first centibillionaire in 1999, when his net worth briefly surpassed US$100 billion. According to Forbes, as of July 2026, his net worth stood at US$106.5 billion, making him the 19th-wealthiest individual in the world.\nBorn and raised in Seattle, Washington, Gates was privately educated at Lakeside Sch

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunker = RecursiveCharacterTextSplitter(chunk_size=300,
                                       chunk_overlap=30)
chunks = chunker.split_documents(docs)
vectorstore = FAISS.from_documents(embedding=embeddings, documents=chunks)
retriever = vectorstore.as_retriever()

In [17]:
def get_hyde_query(query):
    template = (
        "Assume you are an expert. Provide a detailed explanation of '{query}'. "
        "Create a hypothetical answer for that topic."
    )

    system_message_prompt = SystemMessagePromptTemplate.from_template(template)
    chat_prompt = ChatPromptTemplate.from_messages([system_message_prompt])
    messages = chat_prompt.format_messages(query=query)

    response = llm.invoke(messages)
    return response.content

get_hyde_query("Bill gates")

"**Bill Gates: An Overview**\n\nBill Gates, born William Henry Gates III on October 28, 1955, in Seattle, Washington, is a prominent American business magnate, software developer, philanthropist, and author. He co-founded Microsoft Corporation, the world’s largest personal-computer software company, which played a pivotal role in shaping the software industry and the global economy in the late 20th century.\n\n### Early Life and Education\n\nGates showed an early interest in computers and technology. He attended Harvard University in the 1970s, where he pursued a pre-law degree but eventually dropped out to focus on his software development ventures. His passion for programming began in his teenage years when he and his friends developed a program for the school’s computer, developing skills that would become invaluable in his later career.\n\n### Microsoft and the Technology Revolution\n\nIn 1975, Gates co-founded Microsoft with Paul Allen. The company initially focused on developing 

In [20]:
from langchain_classic.chains.hyde.base import HypotheticalDocumentEmbedder
hyde_message = HypotheticalDocumentEmbedder.from_llm(
    llm=llm,
    base_embeddings=embeddings,
    prompt_key="web_search"
)

In [26]:

from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
rag_promt = PromptTemplate.from_template(
"""Please write answer of wuestion based on below context
    context: {context}
    Question: {input}
"""
)
rag_chain = create_stuff_documents_chain(llm, rag_promt)

rag_chain_retrieval = create_retrieval_chain(retriever, rag_chain)
response = rag_chain_retrieval.invoke({"input":"Bill gates"})
print(f"Answer from LLM: {response}")

Answer from LLM: {'input': 'Bill gates', 'context': [Document(id='1ed7959d-61dc-40d9-acd2-908095381779', metadata={'title': 'Family of Bill Gates', 'summary': 'Bill Gates is a prominent American business magnate and philanthropist. He co-founded Microsoft, and he and his former wife Melinda French Gates co-founded the Gates Foundation.', 'source': 'https://en.wikipedia.org/wiki/Family_of_Bill_Gates'}, page_content='Bill Gates is a prominent American business magnate and philanthropist. He co-founded Microsoft, and he and his former wife Melinda French Gates co-founded the Gates Foundation.'), Document(id='2242dbe2-1eca-474a-b57c-5328588f05f3', metadata={'title': 'Bill Gates Sr.', 'summary': 'William Henry Gates II (November 30, 1925 – September 14, 2020), better known as Bill Gates Sr., was an American attorney, philanthropist, and civic leader. He was the founder of the law firm Shidler McBroom & Gates (a predecessor of K&L Gates), and also served as president of both the Seattle King